[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_08/notebook_8_1_text_preprocessing.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Clinical Text Preprocessing

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Parse clinical note sections** using rule-based and regex approaches
2. **Tokenize medical text** with consideration for clinical abbreviations and multi-word terms
3. **Detect negation** to distinguish between present vs. absent conditions
4. **De-identify clinical text** by removing Protected Health Information (PHI)
5. **Apply preprocessing pipelines** to prepare text for downstream NLP tasks

## Clinical Context

Clinical notes contain critical patient information but require careful preprocessing:

**Clinical Scenario**: An NLP system analyzing discharge summaries to identify patients with heart failure must:
- Parse different note sections (History of Present Illness, Assessment, Plan)
- Handle abbreviations ("CHF", "EF", "LVEF")
- Detect negations ("no evidence of heart failure" vs. "heart failure")
- Remove patient identifiers (names, dates, MRNs) before analysis

**Why This Matters**:
- **Negation errors**: False positives if "denies chest pain" is interpreted as "chest pain"
- **Section context**: "pneumonia" in Past Medical History ≠ current pneumonia
- **Privacy violations**: HIPAA requires de-identification before sharing data
- **Abbreviation ambiguity**: "MS" = Multiple Sclerosis, Mitral Stenosis, or Morphine Sulfate?

---

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime, timedelta
from collections import Counter
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported successfully!")

## 1. Generate Synthetic Clinical Notes

We'll create realistic synthetic clinical notes with:
- Standard sections (HPI, PMH, Assessment, Plan)
- Medical abbreviations and terminology
- Negation patterns
- Protected Health Information (PHI)

In [ ]:
def generate_clinical_note(patient_id, has_condition=True):
    """
    Generate a synthetic clinical note with realistic structure.

    Args:
        patient_id: Unique patient identifier
        has_condition: If True, patient has heart failure; if False, explicitly denies it

    Returns:
        Dictionary with note metadata and text
    """
    # Generate synthetic PHI
    first_names = ['John', 'Mary', 'Robert', 'Patricia', 'Michael', 'Jennifer', 'William', 'Linda']
    last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis']

    first_name = np.random.choice(first_names)
    last_name = np.random.choice(last_names)
    mrn = f"{np.random.randint(1000000, 9999999)}"

    # Generate dates
    admission_date = datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 365))
    dob = admission_date - timedelta(days=np.random.randint(50*365, 90*365))

    # Phone number
    phone = f"({np.random.randint(200, 999)}) {np.random.randint(200, 999)}-{np.random.randint(1000, 9999)}"

    # Generate note sections based on condition status
    if has_condition:
        hpi = f"""HISTORY OF PRESENT ILLNESS:
{first_name} {last_name} is a {admission_date.year - dob.year}-year-old patient who presents with worsening
shortness of breath and lower extremity edema over the past 2 weeks. Patient reports dyspnea on exertion
and orthopnea requiring 3 pillows to sleep. Previous diagnosis of CHF with reduced EF (LVEF 35%).
Currently on furosemide 40mg daily and lisinopril 10mg daily. Patient admits to occasional dietary
non-compliance with sodium intake."""

        pmh = """PAST MEDICAL HISTORY:
1. Congestive heart failure (systolic dysfunction)
2. Hypertension
3. Type 2 diabetes mellitus
4. Coronary artery disease s/p MI in 2019
5. No history of stroke or TIA"""

        assessment = f"""ASSESSMENT AND PLAN:
1. Acute on chronic systolic heart failure (NYHA Class III)
   - Patient presents with signs of volume overload
   - BNP elevated at 850 pg/mL (reference <100)
   - Chest X-ray shows pulmonary edema
   - Plan: Increase furosemide to 80mg BID, daily weights, strict I/O monitoring
   - Cardiology consult for consideration of device therapy

2. Hypertension - currently controlled
   - Continue lisinopril, monitor BP closely

3. Diabetes - blood sugars elevated
   - A1C 8.2%, increase metformin dose"""
    else:
        hpi = f"""HISTORY OF PRESENT ILLNESS:
{first_name} {last_name} is a {admission_date.year - dob.year}-year-old patient presenting for routine follow-up.
Patient denies any chest pain, shortness of breath, or lower extremity edema. No dyspnea on exertion.
Patient specifically denies any symptoms of heart failure. Exercise tolerance remains good with regular
walks for 30 minutes daily without difficulty. No orthopnea or paroxysmal nocturnal dyspnea."""

        pmh = """PAST MEDICAL HISTORY:
1. Hypertension - well controlled
2. Type 2 diabetes mellitus
3. Hyperlipidemia
4. No history of heart failure
5. No history of coronary artery disease"""

        assessment = f"""ASSESSMENT AND PLAN:
1. Hypertension - well controlled
   - BP 128/78 today, continue current regimen
   - No evidence of heart failure on exam
   - Normal heart sounds, no murmurs, no peripheral edema

2. Diabetes - improving control
   - A1C down to 6.8%, continue metformin

3. Hyperlipidemia
   - LDL at goal, continue statin therapy"""

    # Combine sections
    note_text = f"""DISCHARGE SUMMARY

Patient Name: {first_name} {last_name}
MRN: {mrn}
DOB: {dob.strftime('%m/%d/%Y')}
Admission Date: {admission_date.strftime('%m/%d/%Y')}
Contact: {phone}

---

{hpi}

{pmh}

{assessment}

---
Electronically signed by Dr. Smith on {admission_date.strftime('%m/%d/%Y')}
"""

    return {
        'patient_id': patient_id,
        'mrn': mrn,
        'name': f"{first_name} {last_name}",
        'dob': dob.strftime('%m/%d/%Y'),
        'admission_date': admission_date.strftime('%m/%d/%Y'),
        'phone': phone,
        'has_heart_failure': has_condition,
        'note_text': note_text
    }

# Generate 20 synthetic notes (10 with HF, 10 without)
notes = []
for i in range(20):
    has_hf = i < 10  # First 10 have heart failure
    note = generate_clinical_note(patient_id=i+1, has_condition=has_hf)
    notes.append(note)

print(f"Generated {len(notes)} synthetic clinical notes")
print(f"\nNotes with heart failure: {sum(n['has_heart_failure'] for n in notes)}")
print(f"Notes without heart failure: {sum(not n['has_heart_failure'] for n in notes)}")

# Display example note
print("\n" + "="*80)
print("EXAMPLE CLINICAL NOTE (Patient with Heart Failure):")
print("="*80)
print(notes[0]['note_text'][:800] + "...\n[truncated]")

## 2. Section Parsing

Clinical notes follow semi-structured formats with standard sections:
- **History of Present Illness (HPI)**: Current symptoms and presentation
- **Past Medical History (PMH)**: Historical diagnoses
- **Assessment and Plan**: Clinical reasoning and treatment plan

**Clinical Importance**:
- "pneumonia" in PMH ≠ active pneumonia
- Assessment section carries more weight for current conditions

In [ ]:
class ClinicalSectionParser:
    """
    Parse clinical notes into sections using regex patterns.
    """

    def __init__(self):
        # Define common section headers (case-insensitive)
        self.section_patterns = [
            ('header', r'^(DISCHARGE SUMMARY|CLINICAL NOTE|PROGRESS NOTE)'),
            ('demographics', r'^(Patient Name:|MRN:|DOB:|Admission Date:|Contact:)'),
            ('hpi', r'^HISTORY OF PRESENT ILLNESS:'),
            ('pmh', r'^PAST MEDICAL HISTORY:'),
            ('medications', r'^MEDICATIONS:'),
            ('allergies', r'^ALLERGIES:'),
            ('physical_exam', r'^PHYSICAL EXAM(?:INATION)?:'),
            ('labs', r'^LAB(?:ORATORY)? (?:RESULTS|DATA):'),
            ('imaging', r'^IMAGING:'),
            ('assessment', r'^ASSESSMENT(?: AND PLAN)?:'),
            ('plan', r'^PLAN:'),
            ('signature', r'^Electronically signed by'),
        ]

    def parse_sections(self, note_text):
        """
        Parse note into sections.

        Args:
            note_text: Raw clinical note text

        Returns:
            Dictionary mapping section names to text content
        """
        sections = {}
        lines = note_text.split('\n')

        current_section = 'unknown'
        current_content = []

        for line in lines:
            # Check if line is a section header
            is_header = False
            for section_name, pattern in self.section_patterns:
                if re.search(pattern, line.strip(), re.IGNORECASE):
                    # Save previous section
                    if current_content:
                        sections[current_section] = '\n'.join(current_content).strip()

                    # Start new section
                    current_section = section_name
                    current_content = []
                    is_header = True
                    break

            # Add line to current section if not a header
            if not is_header and line.strip():
                current_content.append(line)

        # Save final section
        if current_content:
            sections[current_section] = '\n'.join(current_content).strip()

        return sections

# Test section parser
parser = ClinicalSectionParser()
example_sections = parser.parse_sections(notes[0]['note_text'])

print("Detected sections:")
for section_name, content in example_sections.items():
    print(f"\n{'='*60}")
    print(f"Section: {section_name.upper()}")
    print(f"Length: {len(content)} characters")
    print(f"Preview: {content[:150]}...")

## 3. Clinical Tokenization

**Why Clinical Text is Different**:
- **Multi-word terms**: "heart failure", "diabetes mellitus", "myocardial infarction"
- **Abbreviations**: CHF, COPD, MI, LVEF, BNP
- **Measurements**: "40mg", "35%", "850 pg/mL"
- **Section numbers**: "1.", "2.", "3."

Standard tokenizers (splitting on whitespace/punctuation) break clinical terms incorrectly.

In [ ]:
class ClinicalTokenizer:
    """
    Tokenize clinical text while preserving medical terms and abbreviations.
    """

    def __init__(self):
        # Common multi-word clinical terms (should be kept together)
        self.clinical_terms = [
            'heart failure', 'myocardial infarction', 'chest pain',
            'shortness of breath', 'blood pressure', 'diabetes mellitus',
            'coronary artery disease', 'congestive heart failure',
            'past medical history', 'history of present illness',
            'lower extremity', 'pulmonary edema', 'blood sugars',
            'exercise tolerance', 'device therapy'
        ]

        # Common abbreviations (should be recognized as single tokens)
        self.abbreviations = [
            'CHF', 'MI', 'CAD', 'COPD', 'HTN', 'DM', 'EF', 'LVEF',
            'BNP', 'A1C', 'BP', 'HR', 'RR', 'O2', 'IV', 'PO',
            'BID', 'TID', 'QID', 'PRN', 'mg', 'mL', 'pg', 'NYHA'
        ]

    def tokenize(self, text, preserve_clinical_terms=True):
        """
        Tokenize text with special handling for clinical terms.

        Args:
            text: Input text
            preserve_clinical_terms: If True, keep multi-word clinical terms together

        Returns:
            List of tokens
        """
        # Convert to lowercase for processing
        text_lower = text.lower()

        # Replace multi-word clinical terms with underscored versions
        protected_text = text_lower
        if preserve_clinical_terms:
            for term in self.clinical_terms:
                # Replace spaces with underscores to protect term
                protected_term = term.replace(' ', '_')
                protected_text = protected_text.replace(term, protected_term)

        # Tokenize using regex: words, numbers with units, abbreviations
        # Matches: word characters, numbers with optional units (40mg, 35%)
        pattern = r'\b[\w]+[%]?\b'
        tokens = re.findall(pattern, protected_text)

        # Convert underscores back to spaces for clinical terms
        tokens = [token.replace('_', ' ') for token in tokens]

        return tokens

    def tokenize_with_positions(self, text):
        """
        Tokenize and preserve character positions (useful for de-identification).

        Returns:
            List of (token, start_pos, end_pos) tuples
        """
        tokens_with_pos = []
        pattern = r'\b[\w]+[%]?\b'

        for match in re.finditer(pattern, text.lower()):
            tokens_with_pos.append((
                match.group(),
                match.start(),
                match.end()
            ))

        return tokens_with_pos

# Test tokenizer
tokenizer = ClinicalTokenizer()

# Example sentences
test_sentences = [
    "Patient presents with acute heart failure and elevated BNP at 850 pg/mL.",
    "History of myocardial infarction in 2019, currently on furosemide 40mg BID.",
    "No evidence of pulmonary edema on chest X-ray."
]

print("Tokenization Comparison:\n")
for sentence in test_sentences:
    # Simple whitespace tokenization
    simple_tokens = sentence.split()

    # Clinical tokenization
    clinical_tokens = tokenizer.tokenize(sentence)

    print(f"Original: {sentence}")
    print(f"Simple tokenization ({len(simple_tokens)} tokens):")
    print(f"  {simple_tokens[:10]}...")
    print(f"Clinical tokenization ({len(clinical_tokens)} tokens):")
    print(f"  {clinical_tokens[:10]}...")
    print()

# Show token frequency in corpus
all_tokens = []
for note in notes:
    # Only tokenize assessment section (most relevant for current conditions)
    sections = parser.parse_sections(note['note_text'])
    if 'assessment' in sections:
        tokens = tokenizer.tokenize(sections['assessment'])
        all_tokens.extend(tokens)

token_freq = Counter(all_tokens)
print(f"\nTotal tokens in assessment sections: {len(all_tokens)}")
print(f"Unique tokens: {len(token_freq)}")
print(f"\nTop 20 most frequent tokens:")
for token, count in token_freq.most_common(20):
    print(f"  {token:20s} : {count:3d}")

## 4. Negation Detection

**Critical Clinical Challenge**: Distinguishing between:
- ✅ "Patient has heart failure" → Positive finding
- ❌ "No evidence of heart failure" → Negative finding
- ❌ "Patient denies chest pain" → Symptom absent
- ✅ "History of myocardial infarction" → Past condition (positive)

**Algorithm**: NegEx (Chapman et al., 2001)
- Looks for negation triggers before/after medical concepts
- Uses scope windows (typically 5 tokens before/after)

In [ ]:
class NegationDetector:
    """
    Detect negation using NegEx-style algorithm.

    Reference:
    Chapman WW, et al. "A Simple Algorithm for Identifying Negated Findings
    and Diseases in Discharge Summaries." J Biomed Inform. 2001.
    """

    def __init__(self, scope_window=5):
        """
        Args:
            scope_window: Number of tokens to check before/after negation trigger
        """
        self.scope_window = scope_window

        # Pre-negation triggers (come before the concept)
        self.pre_negation_triggers = [
            'no', 'not', 'denies', 'denied', 'without',
            'absence of', 'negative for', 'free of',
            'ruled out', 'rules out', 'no evidence of',
            'no signs of', 'no symptoms of'
        ]

        # Post-negation triggers (come after the concept)
        self.post_negation_triggers = [
            'ruled out', 'is ruled out', 'are ruled out',
            'unlikely', 'was negative'
        ]

        # Pseudo-negation phrases (contain "no" but are not negations)
        self.pseudo_negations = [
            'no change', 'no significant change',
            'not only', 'not only', 'no increase',
            'no decrease'
        ]

        # Termination terms (end negation scope)
        self.terminators = [
            'but', 'however', 'although', 'though',
            'yet', 'except', 'still'
        ]

    def is_negated(self, text, concept, start_pos, end_pos):
        """
        Determine if a concept is negated in the given text.

        Args:
            text: Full text containing the concept
            concept: The medical concept to check
            start_pos: Character position where concept starts
            end_pos: Character position where concept ends

        Returns:
            Tuple: (is_negated: bool, trigger: str or None)
        """
        text_lower = text.lower()

        # Extract window before concept
        window_start = max(0, start_pos - 100)  # ~20 words before
        pre_window = text_lower[window_start:start_pos]

        # Extract window after concept
        window_end = min(len(text), end_pos + 100)  # ~20 words after
        post_window = text_lower[end_pos:window_end]

        # Check for pseudo-negations first (these override negation)
        for pseudo in self.pseudo_negations:
            if pseudo in pre_window or pseudo in post_window:
                return False, None

        # Check for pre-negation triggers
        for trigger in self.pre_negation_triggers:
            if trigger in pre_window:
                # Check if there's a terminator between trigger and concept
                trigger_pos = pre_window.rfind(trigger)
                scope_text = pre_window[trigger_pos:]

                has_terminator = any(term in scope_text for term in self.terminators)

                if not has_terminator:
                    return True, trigger

        # Check for post-negation triggers
        for trigger in self.post_negation_triggers:
            if trigger in post_window:
                # Check if there's a terminator between concept and trigger
                trigger_pos = post_window.find(trigger)
                scope_text = post_window[:trigger_pos]

                has_terminator = any(term in scope_text for term in self.terminators)

                if not has_terminator:
                    return True, trigger

        return False, None

    def find_concepts_with_negation(self, text, concepts):
        """
        Find all occurrences of concepts and check negation status.

        Args:
            text: Text to search
            concepts: List of medical concepts to find

        Returns:
            List of dicts with concept, position, negation status
        """
        results = []
        text_lower = text.lower()

        for concept in concepts:
            concept_lower = concept.lower()

            # Find all occurrences
            start = 0
            while True:
                pos = text_lower.find(concept_lower, start)
                if pos == -1:
                    break

                # Check negation
                is_neg, trigger = self.is_negated(
                    text, concept, pos, pos + len(concept)
                )

                results.append({
                    'concept': concept,
                    'start': pos,
                    'end': pos + len(concept),
                    'negated': is_neg,
                    'negation_trigger': trigger
                })

                start = pos + 1

        return results

# Test negation detection
negation_detector = NegationDetector()

# Test sentences with various negation patterns
test_cases = [
    "Patient has heart failure with reduced ejection fraction.",
    "No evidence of heart failure on physical exam.",
    "Patient denies chest pain or shortness of breath.",
    "Heart failure was ruled out by echocardiography.",
    "History of heart failure, but currently stable.",
    "No change in heart failure status since last visit.",
    "Acute heart failure, not responding to diuretics."
]

concepts_to_find = ['heart failure', 'chest pain', 'shortness of breath']

print("Negation Detection Test Cases:\n")
print("="*80)

for sentence in test_cases:
    print(f"\nSentence: {sentence}")
    results = negation_detector.find_concepts_with_negation(sentence, concepts_to_find)

    if results:
        for result in results:
            status = "NEGATED" if result['negated'] else "AFFIRMED"
            trigger = f" (trigger: '{result['negation_trigger']}')" if result['negated'] else ""
            print(f"  → '{result['concept']}': {status}{trigger}")
    else:
        print(f"  → No target concepts found")

print("\n" + "="*80)

### Apply Negation Detection to Clinical Notes

In [ ]:
# Analyze all notes for heart failure mentions
hf_concepts = ['heart failure', 'CHF', 'congestive heart failure', 'systolic dysfunction']

analysis_results = []

for note in notes:
    # Parse sections
    sections = parser.parse_sections(note['note_text'])

    # Analyze assessment section (most important for current diagnosis)
    if 'assessment' in sections:
        assessment_text = sections['assessment']

        # Find heart failure concepts
        hf_findings = negation_detector.find_concepts_with_negation(
            assessment_text, hf_concepts
        )

        # Determine if patient has heart failure based on findings
        has_hf_affirmed = any(not f['negated'] for f in hf_findings)
        has_hf_negated = any(f['negated'] for f in hf_findings)

        analysis_results.append({
            'patient_id': note['patient_id'],
            'true_label': note['has_heart_failure'],
            'hf_affirmed': has_hf_affirmed,
            'hf_negated': has_hf_negated,
            'num_mentions': len(hf_findings),
            'findings': hf_findings
        })

# Convert to DataFrame for analysis
df_analysis = pd.DataFrame(analysis_results)

# Predict heart failure based on negation analysis
df_analysis['predicted_hf'] = df_analysis['hf_affirmed'] & ~df_analysis['hf_negated']

print("Negation Detection Performance:\n")
print(df_analysis[['patient_id', 'true_label', 'predicted_hf', 'hf_affirmed', 'hf_negated']].head(10))

# Calculate accuracy
accuracy = (df_analysis['true_label'] == df_analysis['predicted_hf']).mean()
print(f"\nAccuracy: {accuracy:.1%}")

# Show confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(df_analysis['true_label'], df_analysis['predicted_hf'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No HF', 'HF'],
            yticklabels=['No HF', 'HF'])
plt.xlabel('Predicted')
plt.ylabel('True Label')
plt.title('Negation Detection Performance\n(Heart Failure Classification)')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(df_analysis['true_label'], df_analysis['predicted_hf'],
                          target_names=['No Heart Failure', 'Heart Failure']))

## 5. De-identification (PHI Removal)

**HIPAA Compliance**: Protected Health Information (PHI) must be removed before data sharing.

**18 HIPAA Identifiers**:
1. Names
2. Geographic subdivisions smaller than state
3. Dates (except year)
4. Phone numbers
5. Fax numbers
6. Email addresses
7. Social Security numbers
8. Medical record numbers
9. Health plan numbers
10. Account numbers
11. Certificate/license numbers
12. Vehicle identifiers
13. Device identifiers
14. URLs
15. IP addresses
16. Biometric identifiers
17. Full-face photos
18. Other unique identifiers

In [ ]:
class ClinicalDeidentifier:
    """
    De-identify clinical notes by removing/masking PHI.

    Note: This is a simplified educational implementation.
    Production systems should use specialized tools like:
    - Philter (MITRE)
    - Amazon Comprehend Medical
    - Microsoft Text Analytics for Health
    """

    def __init__(self, replacement_strategy='mask'):
        """
        Args:
            replacement_strategy: 'mask' (replace with [TYPE]) or 'surrogate' (fake data)
        """
        self.strategy = replacement_strategy

        # Regex patterns for common PHI
        self.patterns = [
            # Dates: MM/DD/YYYY or MM-DD-YYYY
            ('DATE', r'\b\d{1,2}[/-]\d{1,2}[/-]\d{4}\b'),

            # Phone numbers: (XXX) XXX-XXXX or XXX-XXX-XXXX
            ('PHONE', r'\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}'),

            # Medical Record Numbers: 7-8 digits
            ('MRN', r'\bMRN:?\s*\d{7,8}\b'),

            # Ages >= 90 (HIPAA requirement)
            ('AGE', r'\b(9[0-9]|[1-9][0-9]{2,})\s*(?:year|yr)s?(?:-old)?\b'),
        ]

    def deidentify_text(self, text, known_names=None):
        """
        Remove PHI from text.

        Args:
            text: Original text
            known_names: List of patient names to redact (if known)

        Returns:
            De-identified text, list of redactions made
        """
        deidentified = text
        redactions = []

        # Apply regex patterns
        for phi_type, pattern in self.patterns:
            matches = list(re.finditer(pattern, deidentified, re.IGNORECASE))

            # Replace in reverse order to maintain positions
            for match in reversed(matches):
                original = match.group()
                replacement = self._get_replacement(phi_type, original)

                deidentified = (
                    deidentified[:match.start()] +
                    replacement +
                    deidentified[match.end():]
                )

                redactions.append({
                    'type': phi_type,
                    'original': original,
                    'replacement': replacement,
                    'position': match.start()
                })

        # Redact known names
        if known_names:
            for name in known_names:
                # Case-insensitive replacement
                pattern = re.compile(re.escape(name), re.IGNORECASE)
                matches = list(pattern.finditer(deidentified))

                for match in reversed(matches):
                    replacement = self._get_replacement('NAME', name)
                    deidentified = (
                        deidentified[:match.start()] +
                        replacement +
                        deidentified[match.end():]
                    )

                    redactions.append({
                        'type': 'NAME',
                        'original': name,
                        'replacement': replacement,
                        'position': match.start()
                    })

        return deidentified, redactions

    def _get_replacement(self, phi_type, original):
        """
        Generate replacement text based on strategy.
        """
        if self.strategy == 'mask':
            return f"[{phi_type}]"
        elif self.strategy == 'surrogate':
            # Generate fake but realistic replacements
            if phi_type == 'DATE':
                return "01/01/2024"
            elif phi_type == 'PHONE':
                return "(555) 555-5555"
            elif phi_type == 'MRN':
                return "MRN: 0000000"
            elif phi_type == 'AGE':
                return "90+ years old"
            elif phi_type == 'NAME':
                return "[Patient Name]"
            else:
                return f"[{phi_type}]"
        else:
            return "[REDACTED]"

# Test de-identification
deidentifier = ClinicalDeidentifier(replacement_strategy='mask')

# Get example note
example_note = notes[0]
original_text = example_note['note_text']

print("ORIGINAL NOTE (with PHI):")
print("="*80)
print(original_text[:500] + "...\n[truncated]\n")

# De-identify
deidentified_text, redactions = deidentifier.deidentify_text(
    original_text,
    known_names=[example_note['name']]
)

print("\n" + "="*80)
print("DE-IDENTIFIED NOTE:")
print("="*80)
print(deidentified_text[:500] + "...\n[truncated]\n")

print("\n" + "="*80)
print(f"REDACTIONS MADE: {len(redactions)} items")
print("="*80)
for redaction in redactions[:10]:  # Show first 10
    print(f"Type: {redaction['type']:10s} | Original: {redaction['original']:30s} | Replaced with: {redaction['replacement']}")

### Visualize De-identification Coverage

In [ ]:
# Analyze de-identification across all notes
all_redactions = []

for note in notes:
    _, redactions = deidentifier.deidentify_text(
        note['note_text'],
        known_names=[note['name']]
    )
    all_redactions.extend(redactions)

# Count by PHI type
redaction_counts = Counter([r['type'] for r in all_redactions])

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of redaction types
types = list(redaction_counts.keys())
counts = list(redaction_counts.values())

ax1.bar(types, counts, color='steelblue', alpha=0.7)
ax1.set_xlabel('PHI Type')
ax1.set_ylabel('Number of Redactions')
ax1.set_title('De-identification Coverage by PHI Type')
ax1.grid(axis='y', alpha=0.3)

# Pie chart
ax2.pie(counts, labels=types, autopct='%1.1f%%', startangle=90)
ax2.set_title('Distribution of PHI Types Redacted')

plt.tight_layout()
plt.show()

print(f"\nTotal redactions across {len(notes)} notes: {len(all_redactions)}")
print(f"Average redactions per note: {len(all_redactions)/len(notes):.1f}")
print(f"\nBreakdown by PHI type:")
for phi_type, count in redaction_counts.most_common():
    print(f"  {phi_type:10s}: {count:3d} ({count/len(all_redactions)*100:.1f}%)")

## 6. Complete Preprocessing Pipeline

Combine all preprocessing steps into a single pipeline.

In [ ]:
class ClinicalTextPreprocessor:
    """
    Complete preprocessing pipeline for clinical text.
    """

    def __init__(self, deidentify=True, detect_negation=True):
        self.parser = ClinicalSectionParser()
        self.tokenizer = ClinicalTokenizer()
        self.negation_detector = NegationDetector() if detect_negation else None
        self.deidentifier = ClinicalDeidentifier() if deidentify else None

    def preprocess(self, note_text, known_names=None, target_concepts=None):
        """
        Apply complete preprocessing pipeline.

        Args:
            note_text: Raw clinical note
            known_names: Patient names to redact
            target_concepts: Medical concepts to find and check for negation

        Returns:
            Dictionary with preprocessed outputs
        """
        results = {}

        # Step 1: De-identification
        if self.deidentifier:
            note_text, redactions = self.deidentifier.deidentify_text(
                note_text, known_names
            )
            results['deidentified_text'] = note_text
            results['redactions'] = redactions
        else:
            results['deidentified_text'] = note_text
            results['redactions'] = []

        # Step 2: Section parsing
        sections = self.parser.parse_sections(note_text)
        results['sections'] = sections

        # Step 3: Tokenization (for each section)
        results['tokens_by_section'] = {}
        for section_name, section_text in sections.items():
            tokens = self.tokenizer.tokenize(section_text)
            results['tokens_by_section'][section_name] = tokens

        # Step 4: Negation detection (if concepts specified)
        if self.negation_detector and target_concepts:
            results['concept_findings'] = {}
            for section_name, section_text in sections.items():
                findings = self.negation_detector.find_concepts_with_negation(
                    section_text, target_concepts
                )
                if findings:
                    results['concept_findings'][section_name] = findings

        return results

# Test complete pipeline
preprocessor = ClinicalTextPreprocessor(
    deidentify=True,
    detect_negation=True
)

# Process example note
example_note = notes[0]
hf_concepts = ['heart failure', 'CHF', 'systolic dysfunction']

results = preprocessor.preprocess(
    note_text=example_note['note_text'],
    known_names=[example_note['name']],
    target_concepts=hf_concepts
)

print("PREPROCESSING PIPELINE RESULTS")
print("="*80)

print(f"\n1. De-identification: {len(results['redactions'])} PHI items redacted")
print(f"   Types: {', '.join(set(r['type'] for r in results['redactions']))}")

print(f"\n2. Section Parsing: {len(results['sections'])} sections identified")
for section_name in results['sections'].keys():
    num_tokens = len(results['tokens_by_section'].get(section_name, []))
    print(f"   - {section_name:20s}: {num_tokens:4d} tokens")

print(f"\n3. Concept Detection:")
if 'concept_findings' in results:
    for section_name, findings in results['concept_findings'].items():
        print(f"   Section '{section_name}':")
        for finding in findings:
            status = "NEGATED" if finding['negated'] else "AFFIRMED"
            print(f"     - '{finding['concept']}': {status}")
else:
    print("   No target concepts found")

print("\n" + "="*80)

## 7. Real-World Considerations

### Clinical Deployment Challenges

| Challenge | Impact | Solution |
|-----------|--------|----------|
| **Abbreviation ambiguity** | "MS" = Multiple Sclerosis, Mitral Stenosis, Morphine Sulfate | Context-aware disambiguation using section + surrounding terms |
| **Incomplete de-identification** | HIPAA violations, privacy breaches | Use specialized tools (Philter, Amazon Comprehend Medical) + manual review |
| **Copy-paste errors** | Outdated information, false positives | Timestamp validation, change detection |
| **Temporal reasoning** | "resolved pneumonia" vs. "active pneumonia" | Temporal expression extraction, timeline construction |
| **Multi-lingual notes** | English-centric tools fail | Language detection, multi-lingual models |

### Production Best Practices

1. **De-identification**:
   - Use multiple methods (rule-based + ML)
   - Manual review of 100-200 notes
   - Re-identification risk assessment
   - Regular audits

2. **Negation Detection**:
   - NegEx algorithm (rule-based baseline)
   - NegBio (dependency parsing)
   - BERT-based models for complex cases
   - Validate on clinical corpus (i2b2/VA datasets)

3. **Section Classification**:
   - Train on institution-specific templates
   - Handle free-text variations
   - Multi-label classification (sections can overlap)

4. **Validation**:
   - Inter-annotator agreement (κ > 0.80)
   - Clinical expert review
   - Edge case testing (abbreviations, misspellings)

### Common Pitfalls

❌ **Wrong**: Using simple regex for all negation
```python
# FAILS: "no change in heart failure" → incorrectly negates HF
if 'no' in text and 'heart failure' in text:
    negated = True
```

✅ **Correct**: Scope-based negation with pseudo-negation handling
```python
detector = NegationDetector()
is_negated, trigger = detector.is_negated(text, 'heart failure', start, end)
```

---

❌ **Wrong**: Assuming all de-identification is perfect
```python
# Misses: doctor names, facility names, rare PHI
deidentified = simple_regex_deidentification(text)
```

✅ **Correct**: Multi-layered approach + manual review
```python
# Layer 1: Rule-based
text = rule_based_deidentifier.deidentify(text)
# Layer 2: ML-based NER
text = ner_model.deidentify(text)
# Layer 3: Manual review sample
if random.random() < 0.05:  # 5% sample
    queue_for_manual_review(text)
```

## Key Takeaways

### What We Learned

1. **Section Parsing**
   - Clinical notes have semi-structured formats with standard sections
   - Section context is critical: "pneumonia" in PMH ≠ current pneumonia
   - Regex-based parsing works well for structured notes

2. **Clinical Tokenization**
   - Medical terms are often multi-word ("heart failure", "diabetes mellitus")
   - Abbreviations must be preserved as single tokens ("CHF", "LVEF")
   - Measurements need special handling ("40mg", "35%")

3. **Negation Detection**
   - ~10-20% of clinical findings are negated
   - NegEx algorithm: scope-based trigger detection
   - Must handle pseudo-negations ("no change") and terminators ("but")
   - Critical for avoiding false positives in disease detection

4. **De-identification**
   - HIPAA requires removal of 18 identifier types
   - Rule-based approaches miss ~10-20% of PHI
   - Production systems need multi-layered approaches + manual review
   - Trade-off: aggressive de-identification vs. clinical utility

### Connections to Book Chapters

- **Journey 7**: Clinical text parsing for automated coding
- **Chapter 8.2**: Named Entity Recognition builds on tokenization
- **Chapter 8.3**: BERT classification uses preprocessed text
- **Chapter 8.4**: LLM prompting requires section-aware context

### Clinical Validation Requirements

Before deploying in clinical settings:

1. **Negation accuracy** > 95% (validated on clinical corpus)
2. **De-identification recall** > 99% (manual review of 200+ notes)
3. **Section parsing F1** > 0.90 on institution-specific templates
4. **Clinical expert review** of 50+ error cases
5. **Regular audits** (quarterly) for PHI leakage

---

## Exercises

1. **Expand Section Parser**: Add support for "PHYSICAL EXAM", "MEDICATIONS", and "ALLERGIES" sections. Test on your custom notes.

2. **Abbreviation Disambiguation**: Implement a context-aware system to distinguish "MS" = Multiple Sclerosis vs. Mitral Stenosis vs. Morphine Sulfate based on surrounding terms.

3. **Improve Negation Detection**: Add support for double negatives ("not unlikely") and uncertainty phrases ("possible", "suspected").

4. **Enhanced De-identification**: Add patterns for email addresses, URLs, and device serial numbers. Test on notes containing these PHI types.

5. **Temporal Expression Extraction**: Extract and normalize dates/times to identify when conditions started ("2 weeks ago", "since last visit").

6. **Multi-note Analysis**: Implement a patient timeline that tracks condition progression across multiple notes (admission → discharge → follow-up).

7. **Evaluation Metrics**: Create a gold-standard annotated dataset (20 notes) and calculate precision/recall for negation detection and de-identification.

8. **Section Classification ML**: Train a machine learning classifier (Naive Bayes or Logistic Regression) to predict section types based on text features.

---

*This notebook is part of "AI in Healthcare" (Volume 1)*  
*Full implementation available in the complete book companion code*